In [1]:
import os
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder.appName("Tugas11-KopiNusantara").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Path eksplisit file:// supaya Spark membaca disk lokal, bukan HDFS (fs.defaultFS di konfigurasi Hadoop menunjuk ke hdfs://localhost:9000)
path_csv = "file://" + os.path.abspath("tugas11_penjualan.csv")
df = spark.read.csv(path_csv, header=True, inferSchema=True)
df.printSchema()
print("Total baris:", df.count())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/12 11:08:37 WARN Utils: Your hostname, vmi3356457, resolves to a loopback address: 127.0.1.1; using 13.140.149.59 instead (on interface eth0)
26/11/12 11:08:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/zhio/jupyter-env/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/12 11:08:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                          

root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- total_penjualan: double (nullable = true)



[Stage 2:>                                                          (0 + 1) / 1]

Total baris: 2500


## A. Feature Engineering

In [3]:
kolom_kategori = ["nama_menu", "cabang", "waktu_pesan"]

# fit() SEKALI per kolom dan simpan modelnya: dipakai lagi di bagian E
indexer_models = {
    c: StringIndexer(inputCol=c, outputCol=c + "_idx").fit(df)
    for c in kolom_kategori
}

df_indexed = df
for c in kolom_kategori:
    df_indexed = indexer_models[c].transform(df_indexed)

assembler = VectorAssembler(
    inputCols=["qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCol="features")

# Untuk bagian D: assembler kedua yang hanya memakai qty dan harga_satuan
assembler_dasar = VectorAssembler(
    inputCols=["qty", "harga_satuan"],
    outputCol="features_dasar")

df_final = assembler_dasar.transform(assembler.transform(df_indexed))
df_final.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "features", "total_penjualan").show(5, truncate=False)

for c in kolom_kategori:
    print(c, "->", indexer_models[c].labels)

+--------------+----------+-----------+---+------------+-------------------------+------------------+
|nama_menu     |cabang    |waktu_pesan|qty|harga_satuan|features                 |total_penjualan   |
+--------------+----------+-----------+---+------------+-------------------------+------------------+
|Kentang Goreng|Semarang  |Malam      |1  |30000       |[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|Kentang Goreng|Yogyakarta|Sore       |3  |20000       |[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|Kentang Goreng|Semarang  |Siang      |5  |22000       |[5.0,22000.0,5.0,2.0,2.0]|108722.79259154423|
|Kentang Goreng|Yogyakarta|Pagi       |2  |20000       |[2.0,20000.0,5.0,1.0,0.0]|54483.205871792496|
|Roti Bakar    |Magelang  |Malam      |2  |30000       |[2.0,30000.0,4.0,0.0,1.0]|58157.94412015849 |
+--------------+----------+-----------+---+------------+-------------------------+------------------+
only showing top 5 rows
nama_menu -> ['Americano', 'Kopi Susu', 'Cappuccino', 'Es 

## B. Train/Test Split & Pelatihan Model

In [4]:
# Satu split untuk kedua model (B dan D), supaya perbandingannya adil
train, test = df_final.randomSplit([0.75, 0.25], seed=101)
print(f"Training: {train.count()} baris | Testing: {test.count()} baris")

lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model_utama = lr.fit(train)

print("Koefisien :", model_utama.coefficients)
print("Intercept :", model_utama.intercept)

Training: 1839 baris | Testing: 661 baris


Koefisien : [22046.663271025758,3.022920868160672,-123.21425598657399,279.28452461677676,339.6387953231708]
Intercept : -66903.17766485889


## C. Evaluasi Model

In [5]:
pred_utama = model_utama.transform(test)

ev_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
ev_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")

rmse_utama = ev_rmse.evaluate(pred_utama)
r2_utama = ev_r2.evaluate(pred_utama)

print(f"RMSE : Rp {rmse_utama:,.0f}")
print(f"R²   : {r2_utama:.4f}")

RMSE : Rp 10,901
R²   : 0.9107


## D. Eksperimen Perbandingan Fitur

In [6]:
lr_dasar = LinearRegression(featuresCol="features_dasar", labelCol="total_penjualan")
model_dasar = lr_dasar.fit(train)
pred_dasar = model_dasar.transform(test)

rmse_dasar = ev_rmse.evaluate(pred_dasar)
r2_dasar = ev_r2.evaluate(pred_dasar)

print(f"{'Model':<32}{'R²':>10}{'RMSE (Rp)':>14}")
print(f"{'Utama (5 fitur)':<32}{r2_utama:>10.4f}{rmse_utama:>14,.0f}")
print(f"{'Dasar (qty + harga_satuan)':<32}{r2_dasar:>10.4f}{rmse_dasar:>14,.0f}")
print(f"\nSelisih R² (utama - dasar): {r2_utama - r2_dasar:+.4f}")

Model                                   R²     RMSE (Rp)
Utama (5 fitur)                     0.9107        10,901
Dasar (qty + harga_satuan)          0.9108        10,896

Selisih R² (utama - dasar): -0.0001


## E. Prediksi Skenario Baru

In [7]:
# 3 skenario dummy, dibuat TANPA pandas: langsung dari list of tuples
skenario = spark.createDataFrame([
    ("Kopi Susu",  "Magelang",   "Pagi",  2, 18000),
    ("Roti Bakar", "Yogyakarta", "Malam", 4, 25000),
    ("Es Teh",     "Semarang",   "Siang", 5, 15000),
], ["nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan"])

# Pakai StringIndexerModel YANG SAMA dari bagian A (bukan StringIndexer baru)
skenario_idx = skenario
for c in kolom_kategori:
    skenario_idx = indexer_models[c].transform(skenario_idx)

skenario_final = assembler.transform(skenario_idx)

(model_utama.transform(skenario_final)
 .select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction")
 .show(truncate=False))

[Stage 39:======================================>                   (2 + 1) / 3]

+----------+----------+-----------+---+------------+------------------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|prediction        |
+----------+----------+-----------+---+------------+------------------+
|Kopi Susu |Magelang  |Pagi       |2  |18000       |31479.510248098144|
|Roti Bakar|Yogyakarta|Malam      |4  |25000       |96982.5634192546  |
|Es Teh    |Semarang  |Siang      |5  |15000       |89542.15558460016 |
+----------+----------+-----------+---+------------+------------------+



In [ ]:
spark.stop()
print("SparkSession ditutup.")